# Genetic Algorithm to optimize farm layouts with AeroAcoustics

This notebook involves two main parts, the actual GA optimization itself, and then using OpenFAST to validate the results of the best chromosome.

The optimizer uses the vectorized version of the ML AeroAcoustic surrogate interpolator model and FLORIS to replace OpenFAST. 

All of the settings can be found in the following cells.

In [ ]:
from __future__ import annotations

from itertools import product
from pathlib import Path
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import clear_output, display
from matplotlib.colors import Normalize
from matplotlib.patches import Circle, Rectangle
from tqdm.auto import tqdm

from wrapper import (
    AASurrogateMLVec as AASurrogate,
    ParallelFlorisEvaluator,
)

from wrapper.floris import DEFAULT_GCH

from wrapper.farm import (
    Farm,
    AeroAcousticObservers,
)

from wrapper.openfast_base import OpenFASTFile

from wrapper.utils import (
    aggregate_aweighted_oaspl as aggregate_oaspl,
    lout,
    save_metadata,
)

from wrapper.main import run_fastfarm

 # GA

 ## Hyperparameters and configuration

In [ ]:
# -------------------------
# Reproducibility
# -------------------------
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# -------------------------
# GA hyperparameters
# -------------------------
POP_SIZE = 100
GENERATIONS = 50

MUTATION_RATE = 0.35
MUTATION_SPREAD = 100.0

IMMIGRATION_RATE = 0.30
RANDOM_PARENTS = 4

# -------------------------
# Parallel FLORIS
# -------------------------
MAX_WORKERS = 6
FLORIS_CHUNKSIZE = 4

# -------------------------
# Farm / turbine
# -------------------------
NUM_TURBINES = 4

ROTOR_DIAMETER = 126.0
HUB_HEIGHT = 90.0

MIN_SPACING = 3.0 * ROTOR_DIAMETER

MAX_DBA = 35.0
BORDER_SETBACK = 200.0

# -------------------------
# Rectangular land boundary
# -------------------------
X_MIN, X_MAX = 0.0, 3000.0
Y_MIN, Y_MAX = 0.0, 2000.0

# -------------------------
# Penalties
# -------------------------
SPACING_PENALTY_PER_M = 300.0
NOISE_PENALTY_PER_DBA = 500.0

SETBACK_PENALTY_PER_M = 500.0

OUT_OF_BOUNDS_FLAT_PENALTY = 1_000_000.0
OUT_OF_BOUNDS_PENALTY_PER_M = 10_000.0

# "weighted" or "worst_case"
NOISE_PENALTY_MODE = "weighted"

# -------------------------
# Models
# -------------------------
FLORIS_CONFIG_PATH = Path(DEFAULT_GCH).expanduser().resolve()

# If the joblib is located at "dba_model_outputs/best_dba_model"
surrogate = AASurrogate()

# Otherwise use:
# surrogate = AASurrogate(
#     model_path=Path(
#         "dba_model_outputs/best_dba_model.joblib" # <-- point to location
#     ).resolve()
# )

# -------------------------
# OpenFAST / FAST.Farm
# -------------------------
OPENFAST_VERIFY_DIR = Path("openfast_verification")

OPENFAST_VERIFY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OPENFAST_FARM_FILE = Path(r"5MW_farm\base.fstf")

OPENFAST_TURBINE_FILE = Path(r"5MW_farm\base.fst")

OPENFAST_CLEANUP_FILE = Path(r"5MW_farm\cleanup.ps1")

OPENFAST_OBSERVER_Z = 2.0

# High-resolution FAST.Farm turbine box.
HIGH_RES_NUM_POINTS = 33
HIGH_RES_STEP = 5.0

HIGH_RES_BOX_SPAN = (HIGH_RES_NUM_POINTS - 1) * HIGH_RES_STEP

HIGH_RES_BOX_HALF = HIGH_RES_BOX_SPAN / 2.0

# Automatic low-resolution domain settings.
LOW_RES_DX = 10.0
LOW_RES_DY = 10.0
LOW_RES_DZ = 10.0

LOW_RES_UPSTREAM_D = 3.0
LOW_RES_DOWNSTREAM_D = 10.0
LOW_RES_CROSSWIND_D = 4.0
LOW_RES_VERTICAL_D = 1.0

# By default we run ALL scenarios.
OPENFAST_SKIP_EXISTING = True

 ## Weighted wind scenarios







  Speed and direction distributions are treated as independent.







  Joint scenario probability:







      speed_weight * direction_weight

In [ ]:
WIND_SPEEDS = np.array(
    [11.0, 11.5, 12.0],
    dtype=np.float64,
)

WIND_SPEED_WEIGHTS = np.array(
    [0.10, 0.80, 0.10],
    dtype=np.float64,
)

WIND_DIRECTIONS = np.array(
    [260.0, 270.0, 280.0],
    dtype=np.float64,
)

WIND_DIRECTION_WEIGHTS = np.array(
    [0.20, 0.60, 0.20],
    dtype=np.float64,
)

# Used by FLORIS only.
# OpenFAST validation uses WindType=1, so effectively 0% TI.
TURBULENCE_INTENSITY = 0.06

In [ ]:
def _normalized_distribution(
    values,
    weights,
    value_name,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    ).reshape(-1)

    weights = np.asarray(
        weights,
        dtype=np.float64,
    ).reshape(-1)

    if values.size == 0:
        raise ValueError(f"{value_name} cannot be empty.")

    if values.size != weights.size:
        raise ValueError(f"Each {value_name} must have one weight.")

    if np.any(weights < 0.0):
        raise ValueError(f"{value_name} weights cannot be negative.")

    if weights.sum() <= 0.0:
        raise ValueError(f"{value_name} weights must have a positive total.")

    return (
        values,
        weights / weights.sum(),
    )

In [ ]:
def build_wind_scenarios(
    wind_speeds,
    speed_weights,
    wind_directions,
    direction_weights,
):
    speeds, speed_weights = _normalized_distribution(
        wind_speeds,
        speed_weights,
        "wind speed",
    )

    directions, direction_weights = _normalized_distribution(
        wind_directions,
        direction_weights,
        "wind direction",
    )

    scenario_speeds = []
    scenario_directions = []
    scenario_weights = []
    labels = []

    for speed_idx, direction_idx in product(
        range(speeds.size),
        range(directions.size),
    ):
        speed = float(speeds[speed_idx])

        direction = float(directions[direction_idx] % 360.0)

        weight = float(speed_weights[speed_idx] * direction_weights[direction_idx])

        scenario_speeds.append(speed)

        scenario_directions.append(direction)

        scenario_weights.append(weight)

    scenario_weights = np.asarray(
        scenario_weights,
        dtype=np.float64,
    )

    scenario_weights /= scenario_weights.sum()

    tuples = list(
        zip(
            scenario_speeds,
            scenario_directions,
            scenario_weights.tolist(),
        )
    )

    labels = [
        (f"{speed:.1f} m/s, " f"{direction:.0f}°, " f"{weight:.1%}")
        for speed, direction, weight in tuples
    ]

    return {
        "wind_speeds": np.asarray(
            scenario_speeds,
            dtype=np.float64,
        ),
        "wind_directions": np.asarray(
            scenario_directions,
            dtype=np.float64,
        ),
        "weights": scenario_weights,
        "tuples": tuples,
        "labels": labels,
        "speed_values": speeds,
        "speed_weights": speed_weights,
        "direction_values": directions,
        "direction_weights": direction_weights,
    }

In [ ]:
WIND_SCENARIOS = build_wind_scenarios(
    WIND_SPEEDS,
    WIND_SPEED_WEIGHTS,
    WIND_DIRECTIONS,
    WIND_DIRECTION_WEIGHTS,
)

SCENARIO_SPEEDS = WIND_SCENARIOS["wind_speeds"]

SCENARIO_DIRECTIONS = WIND_SCENARIOS["wind_directions"]

SCENARIO_WEIGHTS = WIND_SCENARIOS["weights"]

NUM_SCENARIOS = SCENARIO_WEIGHTS.size

DOMINANT_SCENARIO_INDEX = int(np.argmax(SCENARIO_WEIGHTS))

print(f"Built {NUM_SCENARIOS} scenarios:")

for label in WIND_SCENARIOS["labels"]:
    print(
        "  ",
        label,
    )

print(f"Total weight: " f"{SCENARIO_WEIGHTS.sum():.12f}")

 ## Wind-distribution visualizations

In [ ]:
def plot_wind_scenario_weights(
    scenarios,
):
    speeds = scenarios["speed_values"]

    speed_weights = scenarios["speed_weights"]

    directions = scenarios["direction_values"]

    direction_weights = scenarios["direction_weights"]

    weight_matrix = np.outer(
        speed_weights,
        direction_weights,
    )

    fig, ax = plt.subplots(figsize=(8, 5))

    image = ax.imshow(
        weight_matrix,
        aspect="auto",
    )

    ax.set_xticks(
        np.arange(directions.size),
        [f"{direction:.0f}°" for direction in directions],
    )

    ax.set_yticks(
        np.arange(speeds.size),
        [f"{speed:.1f}" for speed in speeds],
    )

    ax.set_xlabel("Meteorological wind direction")

    ax.set_ylabel("Wind speed [m/s]")

    ax.set_title("Joint wind-scenario weights")

    for i in range(speeds.size):
        for j in range(directions.size):
            ax.text(
                j,
                i,
                f"{weight_matrix[i, j]:.1%}",
                ha="center",
                va="center",
            )

    fig.colorbar(
        image,
        ax=ax,
        label="Scenario probability",
    )

    fig.tight_layout()
    plt.show()

In [ ]:
def plot_wind_direction_distribution(
    scenarios,
):
    directions = scenarios["direction_values"]

    weights = scenarios["direction_weights"]

    angles = np.deg2rad(directions)

    fig = plt.figure(figsize=(7, 7))

    ax = fig.add_subplot(
        111,
        projection="polar",
    )

    ax.set_theta_zero_location("N")

    ax.set_theta_direction(-1)

    bars = ax.bar(
        angles,
        weights,
        width=np.deg2rad(12.0),
        alpha=0.75,
    )

    for bar, weight in zip(
        bars,
        weights,
    ):
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            bar.get_height(),
            f"{weight:.0%}",
            ha="center",
            va="bottom",
        )

    ax.set_title("Wind-direction distribution")

    fig.tight_layout()
    plt.show()

In [ ]:
plot_wind_scenario_weights(WIND_SCENARIOS)

plot_wind_direction_distribution(WIND_SCENARIOS)

 ## GA geometry helpers

In [ ]:
def generate_perimeter_receivers(
    num_per_edge=12,
):
    x_top = np.linspace(
        X_MIN,
        X_MAX,
        num_per_edge,
    )

    x_bottom = np.linspace(
        X_MIN,
        X_MAX,
        num_per_edge,
    )

    y_left = np.linspace(
        Y_MIN,
        Y_MAX,
        num_per_edge,
    )

    y_right = np.linspace(
        Y_MIN,
        Y_MAX,
        num_per_edge,
    )

    return np.column_stack(
        [
            np.concatenate(
                [
                    x_top,
                    x_bottom,
                    np.full(
                        num_per_edge,
                        X_MIN,
                    ),
                    np.full(
                        num_per_edge,
                        X_MAX,
                    ),
                ]
            ),
            np.concatenate(
                [
                    np.full(
                        num_per_edge,
                        Y_MAX,
                    ),
                    np.full(
                        num_per_edge,
                        Y_MIN,
                    ),
                    y_left,
                    y_right,
                ]
            ),
        ]
    )

In [ ]:
receivers = generate_perimeter_receivers()

noise_norm = Normalize(
    vmin=MAX_DBA - 10.0,
    vmax=MAX_DBA + 5.0,
)

noise_cmap = plt.colormaps["viridis"]

In [ ]:
def init_population(
    pop_size,
    num_turbines,
):
    x_coords = rng.uniform(
        X_MIN + BORDER_SETBACK,
        X_MAX - BORDER_SETBACK,
        (
            pop_size,
            num_turbines,
            1,
        ),
    )

    y_coords = rng.uniform(
        Y_MIN + BORDER_SETBACK,
        Y_MAX - BORDER_SETBACK,
        (
            pop_size,
            num_turbines,
            1,
        ),
    )

    return np.concatenate(
        [
            x_coords,
            y_coords,
        ],
        axis=2,
    )

In [ ]:
def crossover(
    parent1,
    parent2,
):
    mask = rng.random(parent1.shape) > 0.5

    return np.where(
        mask,
        parent1,
        parent2,
    )

In [ ]:
def mutate(
    child,
):
    child = child.copy()

    if rng.random() < MUTATION_RATE:
        child += rng.normal(
            0.0,
            MUTATION_SPREAD,
            child.shape,
        )

        child[:, 0] = np.clip(
            child[:, 0],
            X_MIN + BORDER_SETBACK,
            X_MAX - BORDER_SETBACK,
        )

        child[:, 1] = np.clip(
            child[:, 1],
            Y_MIN + BORDER_SETBACK,
            Y_MAX - BORDER_SETBACK,
        )

    return child

 ## Shared acoustic / fitness helpers

In [ ]:
def energetic_weighted_dba(
    levels_dba,
    weights,
    axis,
):
    levels_dba = np.asarray(
        levels_dba,
        dtype=np.float64,
    )

    weights = np.asarray(
        weights,
        dtype=np.float64,
    )

    shape = [1] * levels_dba.ndim

    shape[axis] = weights.size

    weighted_energy = np.sum(
        weights.reshape(shape)
        * np.power(
            10.0,
            levels_dba / 10.0,
        ),
        axis=axis,
    )

    return 10.0 * np.log10(
        np.maximum(
            weighted_energy,
            np.finfo(float).tiny,
        )
    )

In [ ]:
def energetic_mean_dba(
    values,
    axis=0,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    return 10.0 * np.log10(
        np.mean(
            np.power(
                10.0,
                values / 10.0,
            ),
            axis=axis,
        )
    )

In [ ]:
def calculate_layout_penalties(
    layout_2d,
    min_spacing=MIN_SPACING,
    border_setback=BORDER_SETBACK,
    x_min=X_MIN,
    x_max=X_MAX,
    y_min=Y_MIN,
    y_max=Y_MAX,
):
    layout_2d = np.asarray(
        layout_2d,
        dtype=np.float64,
    )

    n_turbines = layout_2d.shape[0]

    # Spacing
    differences = layout_2d[:, None, :] - layout_2d[None, :, :]

    distances = np.linalg.norm(
        differences,
        axis=-1,
    )

    upper = np.triu_indices(
        n_turbines,
        k=1,
    )

    unique_distances = distances[upper]

    spacing_penalty = float(
        np.maximum(
            min_spacing - unique_distances,
            0.0,
        ).sum()
        * SPACING_PENALTY_PER_M
    )

    # Boundary
    x = layout_2d[:, 0]
    y = layout_2d[:, 1]

    border_distances = np.stack(
        (
            x - x_min,
            x_max - x,
            y - y_min,
            y_max - y,
        ),
        axis=-1,
    )

    setback_penalty = float(
        np.maximum(
            border_setback - border_distances,
            0.0,
        ).sum()
        * SETBACK_PENALTY_PER_M
    )

    out_of_bounds_mask = border_distances < 0.0

    if np.any(out_of_bounds_mask):
        out_of_bounds_meters = float(np.abs(border_distances[out_of_bounds_mask]).sum())

        out_of_bounds_penalty = (
            OUT_OF_BOUNDS_FLAT_PENALTY
            + out_of_bounds_meters * OUT_OF_BOUNDS_PENALTY_PER_M
        )

    else:
        out_of_bounds_penalty = 0.0

    border_penalty = setback_penalty + out_of_bounds_penalty

    return {
        "spacing_penalty": spacing_penalty,
        "setback_penalty": setback_penalty,
        "out_of_bounds_penalty": (out_of_bounds_penalty),
        "border_penalty": border_penalty,
    }

 ## Persistent parallel FLORIS evaluator





  The process pool and one FLORIS model per worker remain alive for the entire GA.

In [ ]:
floris_evaluator = ParallelFlorisEvaluator(
    config_path=(FLORIS_CONFIG_PATH),
    turbine_type="nrel_5mw",
    max_workers=MAX_WORKERS,
    chunksize=FLORIS_CHUNKSIZE,
    show_progress=True,
)

 ## Population fitness

In [ ]:
def evaluate_population_fitness(
    population,
    surrogate,
    floris_evaluator,
    receivers,
    *,
    scenario_wind_speeds,
    scenario_wind_directions,
    scenario_weights,
    turbulence_intensity,
    min_spacing,
    max_dba,
    border_setback,
    x_min,
    x_max,
    y_min,
    y_max,
):
    population = np.asarray(
        population,
        dtype=np.float64,
    )

    receivers = np.asarray(
        receivers,
        dtype=np.float64,
    )

    scenario_wind_speeds = np.asarray(
        scenario_wind_speeds,
        dtype=np.float64,
    ).reshape(-1)

    scenario_wind_directions = np.asarray(
        scenario_wind_directions,
        dtype=np.float64,
    ).reshape(-1)

    scenario_weights = np.asarray(
        scenario_weights,
        dtype=np.float64,
    ).reshape(-1)

    scenario_weights /= scenario_weights.sum()

    pop_size, num_turbines, _ = population.shape

    num_scenarios = scenario_weights.size

    # ---------------------------------------------------------
    # 1. Persistent parallel FLORIS
    # ---------------------------------------------------------
    floris_results = floris_evaluator.evaluate_population(
        population,
        wind_speeds=(scenario_wind_speeds),
        wind_directions=(scenario_wind_directions),
        turbulence_intensities=(turbulence_intensity),
        progress_desc=("FLORIS layouts"),
        progress_position=1,
        progress_leave=False,
    )

    scenario_powers_kw = np.asarray(
        floris_results["powers_kw"],
        dtype=np.float64,
    )

    scenario_waked_velocities = np.asarray(
        floris_results["waked_velocities"],
        dtype=np.float64,
    )

    expected_powers_kw = np.sum(
        scenario_powers_kw
        * scenario_weights[
            None,
            :,
        ],
        axis=1,
    )

    # ---------------------------------------------------------
    # 2. Vectorized acoustic surrogate
    # ---------------------------------------------------------
    acoustic_layouts = np.broadcast_to(
        population[:, None, :, :],
        (
            pop_size,
            num_scenarios,
            num_turbines,
            2,
        ),
    ).reshape(
        pop_size * num_scenarios,
        num_turbines,
        2,
    )

    acoustic_velocities = scenario_waked_velocities.reshape(
        pop_size * num_scenarios,
        num_turbines,
    )

    acoustic_directions = np.tile(
        scenario_wind_directions,
        pop_size,
    )

    flat_receiver_dba = surrogate.evaluate_farm_dBA(
        turbine_coords=(acoustic_layouts),
        receiver_coords=receivers,
        turbine_wind_speeds=(acoustic_velocities),
        wind_dir_deg=(acoustic_directions),
        only_aggregate=True,
    )

    scenario_receiver_dba = np.asarray(
        flat_receiver_dba,
        dtype=np.float64,
    ).reshape(
        pop_size,
        num_scenarios,
        receivers.shape[0],
    )

    scenario_max_noises = scenario_receiver_dba.max(axis=2)

    expected_receiver_dba = energetic_weighted_dba(
        scenario_receiver_dba,
        scenario_weights,
        axis=1,
    )

    expected_max_noises = expected_receiver_dba.max(axis=1)

    worst_case_noises = scenario_max_noises.max(axis=1)

    # ---------------------------------------------------------
    # 3. Vectorized spacing penalties
    # ---------------------------------------------------------
    coordinate_differences = population[:, :, None, :] - population[:, None, :, :]

    pairwise_distances = np.linalg.norm(
        coordinate_differences,
        axis=-1,
    )

    upper_triangle = np.triu_indices(
        num_turbines,
        k=1,
    )

    unique_pair_distances = pairwise_distances[
        :,
        upper_triangle[0],
        upper_triangle[1],
    ]

    spacing_penalties = (
        np.maximum(
            min_spacing - unique_pair_distances,
            0.0,
        ).sum(axis=1)
        * SPACING_PENALTY_PER_M
    )

    # ---------------------------------------------------------
    # 4. Noise penalties
    # ---------------------------------------------------------
    scenario_noise_penalties = (
        np.maximum(
            scenario_max_noises - max_dba,
            0.0,
        )
        * NOISE_PENALTY_PER_DBA
    )

    weighted_noise_penalties = np.sum(
        scenario_noise_penalties
        * scenario_weights[
            None,
            :,
        ],
        axis=1,
    )

    worst_case_noise_penalties = (
        np.maximum(
            worst_case_noises - max_dba,
            0.0,
        )
        * NOISE_PENALTY_PER_DBA
    )

    if NOISE_PENALTY_MODE == "weighted":
        noise_penalties = weighted_noise_penalties

    elif NOISE_PENALTY_MODE == "worst_case":
        noise_penalties = worst_case_noise_penalties

    else:
        raise ValueError(
            f"NOISE_PENALTY_MODE must be 'weighted' or 'worst_case', not '{NOISE_PENALTY_MODE}'."
        )

    # ---------------------------------------------------------
    # 5. Boundary penalties
    # ---------------------------------------------------------
    x_coords = population[:, :, 0]

    y_coords = population[:, :, 1]

    all_border_distances = np.stack(
        (
            x_coords - x_min,
            x_max - x_coords,
            y_coords - y_min,
            y_max - y_coords,
        ),
        axis=-1,
    )

    setback_penalties = (
        np.maximum(
            border_setback - all_border_distances,
            0.0,
        ).sum(axis=(1, 2))
        * SETBACK_PENALTY_PER_M
    )

    out_of_bounds_mask = all_border_distances < 0.0

    out_of_bounds_meters = np.where(
        out_of_bounds_mask,
        -all_border_distances,
        0.0,
    ).sum(axis=(1, 2))

    has_out_of_bounds = np.any(
        out_of_bounds_mask,
        axis=(1, 2),
    )

    out_of_bounds_penalties = np.where(
        has_out_of_bounds,
        OUT_OF_BOUNDS_FLAT_PENALTY + out_of_bounds_meters * OUT_OF_BOUNDS_PENALTY_PER_M,
        0.0,
    )

    border_penalties = setback_penalties + out_of_bounds_penalties

    # ---------------------------------------------------------
    # 6. Fitness
    # ---------------------------------------------------------
    fitness_scores = (
        expected_powers_kw - spacing_penalties - noise_penalties - border_penalties
    )

    return {
        "fitness_scores": (fitness_scores),
        "powers_kw": (expected_powers_kw),
        "expected_powers_kw": (expected_powers_kw),
        "scenario_powers_kw": (scenario_powers_kw),
        "max_noises": (worst_case_noises),
        "expected_max_noises": (expected_max_noises),
        "worst_case_noises": (worst_case_noises),
        "receiver_dba": (expected_receiver_dba),
        "scenario_receiver_dba": (scenario_receiver_dba),
        "scenario_max_noises": (scenario_max_noises),
        "waked_velocities": (scenario_waked_velocities),
        "scenario_weights": (scenario_weights),
        "spacing_penalties": (spacing_penalties),
        "noise_penalties": (noise_penalties),
        "weighted_noise_penalties": (weighted_noise_penalties),
        "worst_case_noise_penalties": (worst_case_noise_penalties),
        "border_penalties": (border_penalties),
    }

 ## Wind arrows and live GA visualization

In [ ]:
# Utility to convert FLORIS angle -> OpenFAST angle
def meteorological_direction_to_source_angle(
    wind_direction_deg,
):
    return np.deg2rad(90.0 - wind_direction_deg)

In [ ]:
def ray_rectangle_intersection(
    center_x,
    center_y,
    direction_x,
    direction_y,
    x_min,
    x_max,
    y_min,
    y_max,
):
    candidates = []

    if abs(direction_x) > 1e-12:
        for x_edge in (
            x_min,
            x_max,
        ):
            t = (x_edge - center_x) / direction_x

            if t > 0.0:
                y = center_y + t * direction_y

                if y_min <= y <= y_max:
                    candidates.append(
                        (
                            t,
                            x_edge,
                            y,
                        )
                    )

    if abs(direction_y) > 1e-12:
        for y_edge in (
            y_min,
            y_max,
        ):
            t = (y_edge - center_y) / direction_y

            if t > 0.0:
                x = center_x + t * direction_x

                if x_min <= x <= x_max:
                    candidates.append(
                        (
                            t,
                            x,
                            y_edge,
                        )
                    )

    if not candidates:
        raise RuntimeError("No rectangle intersection.")

    _, boundary_x, boundary_y = min(
        candidates,
        key=lambda item: item[0],
    )

    return (
        boundary_x,
        boundary_y,
    )

In [ ]:
def draw_weighted_wind_arrows(
    ax,
    wind_directions,
    direction_weights,
):
    directions, weights = _normalized_distribution(
        wind_directions,
        direction_weights,
        "wind direction",
    )

    center_x = 0.5 * (X_MIN + X_MAX)

    center_y = 0.5 * (Y_MIN + Y_MAX)

    for direction, weight in zip(
        directions,
        weights,
    ):
        source_angle = meteorological_direction_to_source_angle(direction)

        outward_x = np.cos(source_angle)

        outward_y = np.sin(source_angle)

        start_x, start_y = ray_rectangle_intersection(
            center_x,
            center_y,
            outward_x,
            outward_y,
            X_MIN,
            X_MAX,
            Y_MIN,
            Y_MAX,
        )

        end_fraction = 0.30

        end_x = start_x + end_fraction * (center_x - start_x)

        end_y = start_y + end_fraction * (center_y - start_y)

        ax.annotate(
            "",
            xy=(
                end_x,
                end_y,
            ),
            xytext=(
                start_x,
                start_y,
            ),
            arrowprops={
                "arrowstyle": "-|>",
                "linewidth": (1.0 + 5.0 * weight),
                "alpha": 0.85,
            },
            zorder=7,
        )

        label_x = start_x + 0.12 * (center_x - start_x)

        label_y = start_y + 0.12 * (center_y - start_y)

        ax.annotate(
            (f"{direction:.0f}° " f"({weight:.0%})"),
            xy=(
                label_x,
                label_y,
            ),
            xytext=(
                4,
                4,
            ),
            textcoords=("offset points"),
            fontsize=8,
            fontweight="bold",
            bbox={
                "boxstyle": ("round,pad=0.2"),
                "facecolor": "white",
                "alpha": 0.75,
                "edgecolor": "none",
            },
            zorder=8,
        )

In [ ]:
def update_visualization(
    ax,
    fig,
    gen,
    fitness,
    expected_power,
    worst_noise,
    layout_2d,
    expected_receiver_dba,
    dominant_waked_velocities,
):
    ax.clear()

    boundary = Rectangle(
        (
            X_MIN,
            Y_MIN,
        ),
        X_MAX - X_MIN,
        Y_MAX - Y_MIN,
        edgecolor="gray",
        facecolor="none",
        linewidth=1.5,
        linestyle="--",
    )

    ax.add_patch(boundary)

    for i, (
        x,
        y,
    ) in enumerate(layout_2d):
        ax.add_patch(
            Circle(
                (
                    x,
                    y,
                ),
                radius=(ROTOR_DIAMETER / 2.0),
                alpha=0.15,
                linestyle="--",
                linewidth=1.2,
            )
        )

        ax.annotate(
            (f"{dominant_waked_velocities[i]:.2f} " f"m/s"),
            (
                x,
                y,
            ),
            xytext=(
                10,
                10,
            ),
            textcoords=("offset points"),
            fontsize=8,
            fontweight="bold",
            bbox={
                "boxstyle": ("round,pad=0.2"),
                "facecolor": "white",
                "alpha": 0.7,
                "edgecolor": "none",
            },
        )

    obs_colors = noise_cmap(noise_norm(expected_receiver_dba))

    ax.scatter(
        receivers[:, 0],
        receivers[:, 1],
        c=obs_colors,
        s=25,
    )

    ax.scatter(
        layout_2d[:, 0],
        layout_2d[:, 1],
        marker="x",
        s=120,
        linewidth=1.5,
        label="WT",
    )

    draw_weighted_wind_arrows(
        ax,
        WIND_SCENARIOS["direction_values"],
        WIND_SCENARIOS["direction_weights"],
    )

    dominant_label = WIND_SCENARIOS["labels"][DOMINANT_SCENARIO_INDEX]

    ax.set_xlim(
        X_MIN - 160,
        X_MAX + 160,
    )

    ax.set_ylim(
        Y_MIN - 160,
        Y_MAX + 160,
    )

    ax.set_aspect(
        "equal",
        adjustable="box",
    )

    ax.set_xlabel("X [m]")

    ax.set_ylabel("Y [m]")

    ax.grid(
        True,
        linestyle=":",
        alpha=0.7,
    )

    ax.set_title(
        (
            f"GA Generation "
            f"{gen + 1}/{GENERATIONS}\n"
            f"Fitness: {fitness:.0f} | "
            f"Expected power: "
            f"{expected_power:.1f} kW | "
            f"Worst dBA: "
            f"{worst_noise:.2f}\n"
            f"Velocity labels: "
            f"{dominant_label}"
        )
    )

    fig.canvas.draw_idle()
    fig.canvas.start_event_loop(0.01)

    clear_output(wait=True)

    display(fig)

 ## Genetic algorithm

In [ ]:
def run_ga():
    population = init_population(
        POP_SIZE,
        NUM_TURBINES,
    )

    best_layout_ever = None
    best_fitness_ever = -np.inf

    best_pwr_ever = 0.0
    best_noise_ever = 0.0

    best_receiver_dba_ever = None
    best_velocities_ever = None

    num_immigrants = int(POP_SIZE * IMMIGRATION_RATE)

    cutoff_index = POP_SIZE - num_immigrants

    plt.ion()

    fig, ax = plt.subplots(figsize=(10, 7))

    gen_pbar = tqdm(
        range(GENERATIONS),
        desc="GA optimization",
        position=0,
        leave=True,
    )

    for gen in gen_pbar:
        evaluation = evaluate_population_fitness(
            population=population,
            surrogate=surrogate,
            floris_evaluator=(floris_evaluator),
            receivers=receivers,
            scenario_wind_speeds=(SCENARIO_SPEEDS),
            scenario_wind_directions=(SCENARIO_DIRECTIONS),
            scenario_weights=(SCENARIO_WEIGHTS),
            turbulence_intensity=(TURBULENCE_INTENSITY),
            min_spacing=(MIN_SPACING),
            max_dba=(MAX_DBA),
            border_setback=(BORDER_SETBACK),
            x_min=X_MIN,
            x_max=X_MAX,
            y_min=Y_MIN,
            y_max=Y_MAX,
        )

        fitness_scores = evaluation["fitness_scores"]

        powers = evaluation["expected_powers_kw"]

        worst_noises = evaluation["worst_case_noises"]

        expected_receiver_dba = evaluation["receiver_dba"]

        scenario_waked_velocities = evaluation["waked_velocities"]

        current_best_idx = int(np.argmax(fitness_scores))

        current_best_fit = float(fitness_scores[current_best_idx])

        current_best_velocities = scenario_waked_velocities[
            current_best_idx,
            DOMINANT_SCENARIO_INDEX,
        ]

        if current_best_fit > best_fitness_ever:
            best_fitness_ever = current_best_fit

            best_pwr_ever = float(powers[current_best_idx])

            best_noise_ever = float(worst_noises[current_best_idx])

            best_layout_ever = population[current_best_idx].copy()

            best_receiver_dba_ever = expected_receiver_dba[current_best_idx].copy()

            best_velocities_ever = current_best_velocities.copy()

        gen_pbar.set_postfix(
            {
                "Best fit": (f"{best_fitness_ever:.0f}"),
                "Expected power": (f"{best_pwr_ever:.1f} kW"),
                "Worst dBA": (f"{best_noise_ever:.1f}"),
            }
        )

        update_visualization(
            ax,
            fig,
            gen,
            fitness_scores[current_best_idx],
            powers[current_best_idx],
            worst_noises[current_best_idx],
            population[current_best_idx],
            expected_receiver_dba[current_best_idx],
            current_best_velocities,
        )

        new_population = np.zeros_like(population)

        new_population[0] = best_layout_ever.copy()

        for i in range(
            1,
            cutoff_index,
        ):
            contenders = rng.choice(
                POP_SIZE,
                RANDOM_PARENTS,
                replace=False,
            )

            parent1_idx = contenders[np.argmax(fitness_scores[contenders])]

            contenders = rng.choice(
                POP_SIZE,
                RANDOM_PARENTS,
                replace=False,
            )

            parent2_idx = contenders[np.argmax(fitness_scores[contenders])]

            child = crossover(
                population[parent1_idx],
                population[parent2_idx],
            )

            new_population[i] = mutate(child)

        if num_immigrants > 0:
            new_population[cutoff_index:] = init_population(
                num_immigrants,
                NUM_TURBINES,
            )

        population = new_population

    plt.ioff()

    print(
        "\n[✓] Optimization finished"
        f"\n    Expected power      : "
        f"{best_pwr_ever:.2f} kW"
        f"\n    Worst boundary noise: "
        f"{best_noise_ever:.2f} dBA"
        f"\n    Scenario count      : "
        f"{NUM_SCENARIOS}"
    )

    plt.show()

    return {
        "best_layout": (best_layout_ever),
        "best_fitness": (best_fitness_ever),
        "best_expected_power_kw": (best_pwr_ever),
        "best_worst_noise_dba": (best_noise_ever),
        "best_expected_receiver_dba": (best_receiver_dba_ever),
        "best_dominant_waked_velocities": (best_velocities_ever),
    }

 # Driver



  The persistent FLORIS pool remains alive after this cell so it can be reused for validation.

In [ ]:
results = run_ga()

best_layout = results["best_layout"]

best_pwr = results["best_expected_power_kw"]

best_noise = results["best_worst_noise_dba"]

  # OpenFAST / FAST.Farm validation





  Every scenario in `WIND_SCENARIOS["tuples"]` is run.





  OpenFAST uses deterministic `WindType = 1`, so there is no TurbSim field and the OpenFAST validation assumes 0% turbulence.





  The layout and receiver coordinates are rotated for each meteorological wind direction so OpenFAST / FAST.Farm sees the inflow aligned with its +X frame.

 ## Automatic FAST.Farm low-resolution domain

In [ ]:
def calculate_fastfarm_lowres_domain(
    turbine_positions,
    *,
    rotor_diameter=ROTOR_DIAMETER,
    hub_height=HUB_HEIGHT,
    dx=LOW_RES_DX,
    dy=LOW_RES_DY,
    dz=LOW_RES_DZ,
    upstream_clearance_D=(LOW_RES_UPSTREAM_D),
    downstream_clearance_D=(LOW_RES_DOWNSTREAM_D),
    crosswind_clearance_D=(LOW_RES_CROSSWIND_D),
    vertical_clearance_D=(LOW_RES_VERTICAL_D),
):
    """
    Calculate an automatic FAST.Farm low-resolution domain.

    Assumes the layout has already been rotated such that OpenFAST inflow
    and primary wake advection are along +X.
    """
    positions = np.asarray(
        turbine_positions,
        dtype=np.float64,
    )

    if positions.ndim != 2:
        raise ValueError("turbine_positions must be 2D.")

    if positions.shape[1] == 2:
        positions = np.column_stack(
            (
                positions,
                np.zeros(len(positions)),
            )
        )

    if positions.shape[1] != 3:
        raise ValueError("Expected shape (N,2) or (N,3).")

    D = float(rotor_diameter)

    turbine_x_min = float(positions[:, 0].min())

    turbine_x_max = float(positions[:, 0].max())

    turbine_y_min = float(positions[:, 1].min())

    turbine_y_max = float(positions[:, 1].max())

    desired_x_min = turbine_x_min - upstream_clearance_D * D

    desired_x_max = turbine_x_max + downstream_clearance_D * D

    desired_y_min = turbine_y_min - crosswind_clearance_D * D

    desired_y_max = turbine_y_max + crosswind_clearance_D * D

    desired_z_min = 0.0

    rotor_top = hub_height + D / 2.0

    desired_z_max = rotor_top + vertical_clearance_D * D

    x0_low = float(np.floor(desired_x_min / dx) * dx)

    y0_low = float(np.floor(desired_y_min / dy) * dy)

    z0_low = float(np.floor(desired_z_min / dz) * dz)

    x1_low = float(np.ceil(desired_x_max / dx) * dx)

    y1_low = float(np.ceil(desired_y_max / dy) * dy)

    z1_low = float(np.ceil(desired_z_max / dz) * dz)

    nx_low = int(np.ceil((x1_low - x0_low) / dx)) + 1

    ny_low = int(np.ceil((y1_low - y0_low) / dy)) + 1

    nz_low = int(np.ceil((z1_low - z0_low) / dz)) + 1

    actual_x_max = x0_low + (nx_low - 1) * dx

    actual_y_max = y0_low + (ny_low - 1) * dy

    actual_z_max = z0_low + (nz_low - 1) * dz

    return {
        "X0_Low": x0_low,
        "Y0_Low": y0_low,
        "Z0_Low": z0_low,
        "dX_Low": float(dx),
        "dY_Low": float(dy),
        "dZ_Low": float(dz),
        "NX_Low": nx_low,
        "NY_Low": ny_low,
        "NZ_Low": nz_low,
        "X_max": float(actual_x_max),
        "Y_max": float(actual_y_max),
        "Z_max": float(actual_z_max),
        "x_span": float(actual_x_max - x0_low),
        "y_span": float(actual_y_max - y0_low),
        "z_span": float(actual_z_max - z0_low),
    }

 ## Optional domain preview for every scenario

In [ ]:
from wrapper.utils import rotate_layout_for_openfast

domain_preview_rows = []

for (
    wind_speed,
    wind_direction,
    scenario_weight,
) in WIND_SCENARIOS["tuples"]:
    rotated_layout, _ = rotate_layout_for_openfast(
        best_layout,
        receivers,
        angle=wind_direction,
    )

    turbine_positions_preview = np.column_stack(
        (
            np.asarray(
                rotated_layout,
                dtype=np.float64,
            ),
            np.zeros(len(rotated_layout)),
        )
    )

    domain = calculate_fastfarm_lowres_domain(turbine_positions_preview)

    domain_preview_rows.append(
        {
            "wind_speed_mps": (wind_speed),
            "direction_deg": (wind_direction),
            "weight": (scenario_weight),
            "X0_Low": (domain["X0_Low"]),
            "X_max": (domain["X_max"]),
            "NX_Low": (domain["NX_Low"]),
            "Y0_Low": (domain["Y0_Low"]),
            "Y_max": (domain["Y_max"]),
            "NY_Low": (domain["NY_Low"]),
            "Z0_Low": (domain["Z0_Low"]),
            "Z_max": (domain["Z_max"]),
            "NZ_Low": (domain["NZ_Low"]),
        }
    )

domain_preview_df = pd.DataFrame(domain_preview_rows)

display(domain_preview_df)

 ## Run one FAST.Farm verification scenario

In [ ]:
def run_openfast_verification_case(
    layout_2d,
    receiver_coords,
    wind_speed,
    wind_direction,
    weight,
    output_root=(OPENFAST_VERIFY_DIR),
    skip_existing=(OPENFAST_SKIP_EXISTING),
):
    rotated_layout, rotated_receivers = rotate_layout_for_openfast(
        layout_2d,
        receiver_coords,
        angle=wind_direction,
    )

    rotated_layout = np.asarray(
        rotated_layout,
        dtype=np.float64,
    )

    rotated_receivers = np.asarray(
        rotated_receivers,
        dtype=np.float64,
    )

    n_turbines = rotated_layout.shape[0]

    turbine_positions = np.column_stack(
        (
            rotated_layout,
            np.zeros(n_turbines),
        )
    )

    observer_positions = np.column_stack(
        (
            rotated_receivers,
            np.full(
                rotated_receivers.shape[0],
                OPENFAST_OBSERVER_Z,
            ),
        )
    )

    run_name = f"verify_" f"{wind_speed:.1f}mps_" f"{wind_direction:.0f}deg"

    folder_path = Path(output_root) / run_name

    if skip_existing and folder_path.is_dir() and any(folder_path.iterdir()):
        tqdm.write(f"Skipping completed case: " f"{folder_path}")

        return {
            "run_name": run_name,
            "folder": str(folder_path),
            "wind_speed": float(wind_speed),
            "wind_direction": float(wind_direction),
            "weight": float(weight),
            "skipped": True,
        }

    subprocess.run(
        [
            "powershell",
            "-ExecutionPolicy",
            "Bypass",
            "-File",
            str(OPENFAST_CLEANUP_FILE),
            "-Force",
        ],
        check=True,
        stdout=(subprocess.DEVNULL),
    )

    farm = Farm(
        str(OPENFAST_FARM_FILE),
        "verification",
    )

    discretization_grid = [
        HIGH_RES_STEP,
        HIGH_RES_STEP,
        HIGH_RES_STEP,
    ]

    # ---------------------------------------------------------
    # Turbines and AA observers
    # ---------------------------------------------------------
    for i in range(n_turbines):
        wt = OpenFASTFile(
            str(OPENFAST_TURBINE_FILE),
            f"WT{i}",
        )

        aero = wt.AeroFile.open()

        aa = aero.AA_InputFile.open()

        aa_locations = AeroAcousticObservers(turbine_positions[i])

        aa_locations.addObservers(observer_positions, True)

        aa_locations.linkToAA(aa)

        aa.toFile()

        aero.AA_InputFile.link(aa)

        aero.toFile()

        wt.AeroFile.link(aero)

        wt.toFile()

        x0_high = (
            turbine_positions[
                i,
                0,
            ]
            - HIGH_RES_BOX_HALF
        )

        y0_high = (
            turbine_positions[
                i,
                1,
            ]
            - HIGH_RES_BOX_HALF
        )

        z0_high = HUB_HEIGHT - HIGH_RES_BOX_HALF

        farm.addWT(
            pos=(turbine_positions[i]),
            turbine=wt,
            high_res_origin=[
                x0_high,
                y0_high,
                z0_high,
            ],
            high_res_grid=(discretization_grid),
        )

    farm = farm.toOFF()

    # ---------------------------------------------------------
    # Automatic low-resolution domain
    # ---------------------------------------------------------
    lowres = calculate_fastfarm_lowres_domain(turbine_positions)

    farm.X0_Low = lowres["X0_Low"]

    farm.Y0_Low = lowres["Y0_Low"]

    farm.Z0_Low = lowres["Z0_Low"]

    farm.dX_Low = lowres["dX_Low"]

    farm.dY_Low = lowres["dY_Low"]

    farm.dZ_Low = lowres["dZ_Low"]

    farm.NX_Low = lowres["NX_Low"]

    farm.NY_Low = lowres["NY_Low"]

    farm.NZ_Low = lowres["NZ_Low"]

    farm.NX_High = HIGH_RES_NUM_POINTS

    farm.NY_High = HIGH_RES_NUM_POINTS

    farm.NZ_High = HIGH_RES_NUM_POINTS

    tqdm.write(
        (
            f"\n{run_name}"
            f"\n  X: "
            f"{lowres['X0_Low']:.0f} "
            f"→ "
            f"{lowres['X_max']:.0f} m "
            f"({lowres['NX_Low']} pts)"
            f"\n  Y: "
            f"{lowres['Y0_Low']:.0f} "
            f"→ "
            f"{lowres['Y_max']:.0f} m "
            f"({lowres['NY_Low']} pts)"
            f"\n  Z: "
            f"{lowres['Z0_Low']:.0f} "
            f"→ "
            f"{lowres['Z_max']:.0f} m "
            f"({lowres['NZ_Low']} pts)"
        )
    )

    # ---------------------------------------------------------
    # Uniform deterministic wind.
    # No TurbSim -> 0% TI in validation.
    # ---------------------------------------------------------
    inflow = farm.InflowFile.open()

    inflow.WindType = 1  # steady

    inflow.HWindSpeed = float(wind_speed)

    # # If you generated a TurbSim file, use as such:
    # inflow.WindType = 3 # binary TurbSim FF
    # inflow.FileName_BTS = "string path..."
    # # If you do plan to use it though, ensure the domain is large enough
    # # We did not use TurbSim files due to the amount of time needed to generate them

    inflow.toFile()

    farm.InflowFile.link(inflow)

    farm.toFile()

    # ---------------------------------------------------------
    # Run FAST.Farm
    # ---------------------------------------------------------
    sim_duration = float(farm["TMax"])

    with tqdm(
        total=sim_duration,
        desc=(f"{wind_speed:.1f} m/s | " f"{wind_direction:.0f}°"),
        leave=False,
    ) as simulation_bar:

        def progress_callback(
            current_time,
            total_time,
        ):
            simulation_bar.n = float(current_time)

            simulation_bar.refresh()

        run_fastfarm(
            farm,
            str(output_root),
            run_name,
            progress_callback,
        )

    # ---------------------------------------------------------
    # Metadata
    # ---------------------------------------------------------
    save_metadata(
        str(output_root),
        run_name,
        wt_pos=(turbine_positions),
        obs_pos=(observer_positions),
        speed=float(wind_speed),
        wind_direction=float(wind_direction),
        scenario_weight=float(weight),
    )

    return {
        "run_name": run_name,
        "folder": str(folder_path),
        "wind_speed": float(wind_speed),
        "wind_direction": float(wind_direction),
        "weight": float(weight),
        "skipped": False,
        **lowres,
    }

 ## Run EVERY OpenFAST wind scenario

In [ ]:
verification_scenarios = list(WIND_SCENARIOS["tuples"])

print(f"OpenFAST verification scenarios: " f"{len(verification_scenarios)}")

print(f"Total scenario weight: " f"{sum(s[2] for s in verification_scenarios):.12f}")

In [ ]:
verification_results = []

for (
    wind_speed,
    wind_direction,
    scenario_weight,
) in tqdm(
    verification_scenarios,
    desc="OpenFAST scenarios",
):
    result = run_openfast_verification_case(
        layout_2d=best_layout,
        receiver_coords=receivers,
        wind_speed=wind_speed,
        wind_direction=(wind_direction),
        weight=(scenario_weight),
    )

    verification_results.append(result)

verification_results_df = pd.DataFrame(verification_results)

display(verification_results_df)

 ## Read OpenFAST steady-state acoustic results

In [ ]:
def get_openfast_steady_state_dba(
    folder_path,
    glob="*AD.AA1.out",
    reducer=energetic_mean_dba,
    steady_fraction=0.5,
    time_column=None,
):
    folder_path = Path(folder_path)

    df = aggregate_oaspl(
        folder_path,
        glob=glob,
    )

    if df is None or len(df) == 0:
        raise ValueError(f"No acoustic results in " f"{folder_path}")

    if not (0.0 < steady_fraction <= 1.0):
        raise ValueError("steady_fraction must be " "> 0 and <= 1.")

    start_index = int(np.floor(len(df) * (1.0 - steady_fraction)))

    steady = df.iloc[start_index:].copy()

    numeric_columns = steady.select_dtypes(include=np.number).columns.tolist()

    if not numeric_columns:
        raise ValueError("No numeric columns found.")

    if time_column is not None:
        acoustic_columns = [
            column for column in numeric_columns if column != time_column
        ]

    else:
        time_like_names = {
            "time",
            "t",
            "time_s",
            "time_[s]",
            "time (s)",
            "timesec",
        }

        detected_time = None

        for column in numeric_columns:
            if str(column).strip().lower() in time_like_names:
                detected_time = column
                break

        if detected_time is None:
            acoustic_columns = numeric_columns

        else:
            acoustic_columns = [
                column for column in numeric_columns if column != detected_time
            ]

    acoustic_values = steady[acoustic_columns].to_numpy(dtype=np.float64)

    receiver_dba = np.asarray(
        reducer(
            acoustic_values,
            axis=0,
        ),
        dtype=np.float64,
    ).reshape(-1)

    return {
        "dataframe": df,
        "steady_state": steady,
        "acoustic_columns": (acoustic_columns),
        "receiver_dba": (receiver_dba),
        "max_dba": float(receiver_dba.max()),
    }

 ## Read OpenFAST steady-state power results

  Each non-AeroAcoustics `.out` file in a scenario folder is inspected
  for the `GenPwr (kW)` channel. The latter half of each turbine output
  is treated as steady state and averaged in linear kW units.

In [ ]:
def get_openfast_steady_state_power_kw(
    folder_path,
    steady_fraction=0.5,
    reducer=np.mean,
    expected_num_turbines=None,
):
    """
    Read steady-state generator power from the OpenFAST turbine output
    files in one FAST.Farm validation scenario.

    Every ``*.out`` file other than ``*.AD.AA1.out`` is inspected. Files
    containing the ``GenPwr (kW)`` channel are treated as turbine OpenFAST
    outputs.

    Returns
    -------
    dict
        turbine_powers_kw:
            Mean/reduced steady-state generator power for each turbine.
        total_power_kw:
            Sum of the steady-state turbine powers for the scenario.
        turbine_files:
            Output files used for the calculation.
        steady_state_frames:
            Back-half DataFrames used for the reduction.
    """
    folder_path = Path(folder_path)

    if not (0.0 < steady_fraction <= 1.0):
        raise ValueError("steady_fraction must be > 0 and <= 1.")

    candidate_files = sorted(
        filepath
        for filepath in folder_path.glob("*.out")
        if not filepath.name.endswith(".AD.AA1.out")
    )

    if not candidate_files:
        raise ValueError(
            f"No non-AeroAcoustics .out files found in {folder_path}"
        )

    turbine_powers_kw = []
    turbine_files = []
    steady_state_frames = []

    for filepath in candidate_files:
        df = lout(str(filepath))

        # FAST.Farm or other output files may also use .out. Only the
        # per-turbine OpenFAST outputs containing GenPwr are used here.
        if "GenPwr (kW)" not in df.columns:
            continue

        if len(df) == 0:
            raise ValueError(f"OpenFAST output is empty: {filepath}")

        start_index = int(
            np.floor(
                len(df)
                * (1.0 - steady_fraction)
            )
        )

        steady = df.iloc[start_index:].copy()

        power_values_kw = steady["GenPwr (kW)"].to_numpy(
            dtype=np.float64
        )

        turbine_power_kw = float(
            reducer(
                power_values_kw,
                axis=0,
            )
        )

        turbine_files.append(filepath)
        steady_state_frames.append(steady)
        turbine_powers_kw.append(turbine_power_kw)

    if not turbine_powers_kw:
        raise ValueError(
            f"No .out files containing 'GenPwr (kW)' found in {folder_path}"
        )

    turbine_powers_kw = np.asarray(
        turbine_powers_kw,
        dtype=np.float64,
    )

    if (
        expected_num_turbines is not None
        and turbine_powers_kw.size != expected_num_turbines
    ):
        raise ValueError(
            f"{folder_path}: found {turbine_powers_kw.size} turbine power "
            f"outputs containing 'GenPwr (kW)', expected "
            f"{expected_num_turbines}. Files used: "
            f"{[path.name for path in turbine_files]}"
        )

    return {
        "turbine_powers_kw": turbine_powers_kw,
        "total_power_kw": float(turbine_powers_kw.sum()),
        "turbine_files": turbine_files,
        "steady_state_frames": steady_state_frames,
    }

 ## Load all OpenFAST scenarios



  Change `OPENFAST_REDUCER` to `np.mean`, `np.max`, `np.median`, or any callable accepting `(values, axis=...)`.

In [ ]:
OPENFAST_REDUCER = energetic_mean_dba

openfast_receiver_dba_list = []
openfast_max_dba = []
openfast_full_results = []

openfast_turbine_powers_kw_list = []
openfast_scenario_powers_kw_list = []
openfast_power_full_results = []

for (
    wind_speed,
    wind_direction,
    scenario_weight,
) in tqdm(
    verification_scenarios,
    desc="Reading OpenFAST validation outputs",
):
    run_name = f"verify_{wind_speed:.1f}mps_{wind_direction:.0f}deg"

    folder_path = OPENFAST_VERIFY_DIR / run_name

    acoustic = get_openfast_steady_state_dba(
        folder_path,
        reducer=(OPENFAST_REDUCER),
        steady_fraction=0.5,
    )

    power = get_openfast_steady_state_power_kw(
        folder_path,
        steady_fraction=0.5,
        reducer=np.mean,
        expected_num_turbines=NUM_TURBINES,
    )

    if acoustic["receiver_dba"].size != len(receivers):
        raise ValueError(
            f"{run_name}: "
            f"OpenFAST returned "
            f"{acoustic['receiver_dba'].size} "
            f"receiver values, expected "
            f"{len(receivers)}."
        )

    openfast_receiver_dba_list.append(acoustic["receiver_dba"])

    openfast_max_dba.append(acoustic["max_dba"])

    openfast_full_results.append(acoustic)

    openfast_turbine_powers_kw_list.append(power["turbine_powers_kw"])

    openfast_scenario_powers_kw_list.append(power["total_power_kw"])

    openfast_power_full_results.append(power)

openfast_receiver_dba = np.stack(
    openfast_receiver_dba_list,
    axis=0,
)

openfast_max_dba = np.asarray(
    openfast_max_dba,
    dtype=np.float64,
)

openfast_turbine_powers_kw = np.stack(
    openfast_turbine_powers_kw_list,
    axis=0,
)

openfast_scenario_powers_kw = np.asarray(
    openfast_scenario_powers_kw_list,
    dtype=np.float64,
)

 ## Re-run best layout through FLORIS for surrogate wake velocities

  FLORIS is re-run only to reproduce the wake-adjusted turbine velocities
  supplied to the ML aeroacoustic surrogate during GA evaluation.
  Validation power is read directly from the OpenFAST turbine outputs.

In [ ]:
best_layout_batch = np.asarray(
    best_layout,
    dtype=np.float64,
)[
    None,
    :,
    :,
]

floris_validation = floris_evaluator.evaluate_population(
    best_layout_batch,
    wind_speeds=(SCENARIO_SPEEDS),
    wind_directions=(SCENARIO_DIRECTIONS),
    turbulence_intensities=(TURBULENCE_INTENSITY),
    progress_desc=("FLORIS best-layout validation"),
    progress_position=0,
    progress_leave=True,
)

floris_scenario_velocities = np.asarray(
    floris_validation["waked_velocities"],
    dtype=np.float64,
)[0]

In [ ]:
surrogate_layout_batch = np.broadcast_to(
    np.asarray(
        best_layout,
        dtype=np.float64,
    )[
        None,
        :,
        :,
    ],
    (
        NUM_SCENARIOS,
        NUM_TURBINES,
        2,
    ),
).copy()

surrogate_receiver_dba = surrogate.evaluate_farm_dBA(
    turbine_coords=(surrogate_layout_batch),
    receiver_coords=receivers,
    turbine_wind_speeds=(floris_scenario_velocities),
    wind_dir_deg=(SCENARIO_DIRECTIONS),
    only_aggregate=True,
)

surrogate_receiver_dba = np.asarray(
    surrogate_receiver_dba,
    dtype=np.float64,
)

surrogate_max_dba = surrogate_receiver_dba.max(axis=1)

  # Full weighted fitness: surrogate vs OpenFAST

  OpenFAST power is held constant in both validation-basis comparisons.

  The only difference between these two values is the acoustic prediction:

  - surrogate validation basis -> OpenFAST power + ML surrogate dBA

  - OpenFAST-validated fitness -> OpenFAST power + OpenFAST dBA

In [ ]:
layout_penalties = calculate_layout_penalties(best_layout)

spacing_penalty = layout_penalties["spacing_penalty"]

border_penalty = layout_penalties["border_penalty"]

expected_power_kw = float(np.sum(openfast_scenario_powers_kw * SCENARIO_WEIGHTS))

surrogate_scenario_noise_penalty = (
    np.maximum(
        surrogate_max_dba - MAX_DBA,
        0.0,
    )
    * NOISE_PENALTY_PER_DBA
)

openfast_scenario_noise_penalty = (
    np.maximum(
        openfast_max_dba - MAX_DBA,
        0.0,
    )
    * NOISE_PENALTY_PER_DBA
)

surrogate_weighted_noise_penalty = float(
    np.sum(surrogate_scenario_noise_penalty * SCENARIO_WEIGHTS)
)

openfast_weighted_noise_penalty = float(
    np.sum(openfast_scenario_noise_penalty * SCENARIO_WEIGHTS)
)

surrogate_worst_noise_penalty = float(surrogate_scenario_noise_penalty.max())

openfast_worst_noise_penalty = float(openfast_scenario_noise_penalty.max())

if NOISE_PENALTY_MODE == "weighted":
    surrogate_final_noise_penalty = surrogate_weighted_noise_penalty

    openfast_final_noise_penalty = openfast_weighted_noise_penalty

elif NOISE_PENALTY_MODE == "worst_case":
    surrogate_final_noise_penalty = surrogate_worst_noise_penalty

    openfast_final_noise_penalty = openfast_worst_noise_penalty

else:
    raise ValueError("Unknown NOISE_PENALTY_MODE.")

surrogate_validated_fitness_basis = (
    expected_power_kw - spacing_penalty - border_penalty - surrogate_final_noise_penalty
)

openfast_validated_fitness = (
    expected_power_kw - spacing_penalty - border_penalty - openfast_final_noise_penalty
)

In [ ]:
print(f"Expected OpenFAST power       : " f"{expected_power_kw:.2f} kW")

print(f"Spacing penalty              : " f"{spacing_penalty:.2f}")

print(f"Border penalty               : " f"{border_penalty:.2f}")

print(f"Noise penalty mode           : " f"{NOISE_PENALTY_MODE}")

print(f"Surrogate noise penalty      : " f"{surrogate_final_noise_penalty:.2f}")

print(f"OpenFAST noise penalty       : " f"{openfast_final_noise_penalty:.2f}")

print(f"Surrogate fitness            : " f"{surrogate_validated_fitness_basis:.2f}")

print(f"OpenFAST-validated fitness   : " f"{openfast_validated_fitness:.2f}")

 ## Scenario comparison table

In [ ]:
receiver_errors = surrogate_receiver_dba - openfast_receiver_dba

scenario_receiver_mae = np.mean(
    np.abs(receiver_errors),
    axis=1,
)

scenario_receiver_rmse = np.sqrt(
    np.mean(
        receiver_errors**2,
        axis=1,
    )
)

scenario_receiver_bias = np.mean(
    receiver_errors,
    axis=1,
)

scenario_max_error_dba = surrogate_max_dba - openfast_max_dba

comparison_df = pd.DataFrame(
    {
        "wind_speed_mps": (SCENARIO_SPEEDS),
        "wind_direction_deg": (SCENARIO_DIRECTIONS),
        "weight": (SCENARIO_WEIGHTS),
        "openfast_power_kw": (openfast_scenario_powers_kw),
        "surrogate_max_dba": (surrogate_max_dba),
        "openfast_max_dba": (openfast_max_dba),
        "max_dba_error": (scenario_max_error_dba),
        "receiver_mae_dba": (scenario_receiver_mae),
        "receiver_rmse_dba": (scenario_receiver_rmse),
        "receiver_bias_dba": (scenario_receiver_bias),
        "surrogate_noise_penalty": (surrogate_scenario_noise_penalty),
        "openfast_noise_penalty": (openfast_scenario_noise_penalty),
    }
)

for turbine_index in range(NUM_TURBINES):
    comparison_df[
        f"openfast_WT{turbine_index + 1}_power_kw"
    ] = openfast_turbine_powers_kw[:, turbine_index]

display(comparison_df)

 ## Weighted validation statistics

In [ ]:
weighted_receiver_mae = float(np.sum(scenario_receiver_mae * SCENARIO_WEIGHTS))

weighted_receiver_rmse = float(
    np.sqrt(np.sum((scenario_receiver_rmse**2) * SCENARIO_WEIGHTS))
)

weighted_receiver_bias = float(np.sum(scenario_receiver_bias * SCENARIO_WEIGHTS))

weighted_max_dba_error = float(np.sum(scenario_max_error_dba * SCENARIO_WEIGHTS))

print(f"Weighted receiver MAE      : " f"{weighted_receiver_mae:.3f} dBA")

print(f"Weighted receiver RMSE     : " f"{weighted_receiver_rmse:.3f} dBA")

print(f"Weighted receiver bias     : " f"{weighted_receiver_bias:+.3f} dBA")

print(f"Weighted maximum-dBA error : " f"{weighted_max_dba_error:+.3f} dBA")

 # Viz

 ## Validation visualizations

 ### Maximum receiver dBA by scenario

In [ ]:
scenario_labels = [
    (f"{speed:.1f} m/s\n" f"{direction:.0f}°\n" f"{weight:.0%}")
    for (
        speed,
        direction,
        weight,
    ) in WIND_SCENARIOS["tuples"]
]

x = np.arange(NUM_SCENARIOS)

bar_width = 0.38

fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(
    x - bar_width / 2,
    surrogate_max_dba,
    width=bar_width,
    label="ML surrogate",
)

ax.bar(
    x + bar_width / 2,
    openfast_max_dba,
    width=bar_width,
    label="OpenFAST",
)

ax.axhline(
    MAX_DBA,
    linestyle="--",
    linewidth=1.5,
    label=(f"Noise limit " f"({MAX_DBA:.1f} dBA)"),
)

ax.set_xticks(x)

ax.set_xticklabels(scenario_labels)

ax.set_ylabel("Maximum receiver level [dBA]")

ax.set_xlabel("Wind scenario")

ax.set_title("Maximum Boundary Noise: " "Surrogate vs OpenFAST")

ax.grid(
    axis="y",
    linestyle=":",
    alpha=0.5,
)

ax.legend()

plt.tight_layout()
plt.show()

 ### Receiver-level parity plot

In [ ]:
surrogate_flat = surrogate_receiver_dba.ravel()

openfast_flat = openfast_receiver_dba.ravel()

minimum_level = float(
    min(
        surrogate_flat.min(),
        openfast_flat.min(),
    )
)

maximum_level = float(
    max(
        surrogate_flat.max(),
        openfast_flat.max(),
    )
)

fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(
    openfast_flat,
    surrogate_flat,
    alpha=0.45,
    s=28,
)

ax.plot(
    [
        minimum_level,
        maximum_level,
    ],
    [
        minimum_level,
        maximum_level,
    ],
    linestyle="--",
    linewidth=1.5,
)

ax.set_xlabel("OpenFAST receiver dBA")

ax.set_ylabel("Surrogate receiver dBA")

ax.set_title("Receiver-Level Acoustic Validation")

ax.grid(
    True,
    linestyle=":",
    alpha=0.5,
)

summary_text = (
    f"Weighted MAE: "
    f"{weighted_receiver_mae:.3f} dBA\n"
    f"Weighted RMSE: "
    f"{weighted_receiver_rmse:.3f} dBA\n"
    f"Weighted bias: "
    f"{weighted_receiver_bias:+.3f} dBA"
)

ax.text(
    0.04,
    0.96,
    summary_text,
    transform=(ax.transAxes),
    va="top",
    bbox={
        "boxstyle": ("round,pad=0.4"),
        "facecolor": "white",
        "alpha": 0.8,
    },
)

plt.tight_layout()
plt.show()

  ## Scenario receiver-error spatial maps



  Positive = surrogate louder than OpenFAST.

  Negative = surrogate quieter than OpenFAST.

In [ ]:
receiver_coords_array = np.asarray(
    receivers,
    dtype=np.float64,
)

absolute_error_limit = float(np.max(np.abs(receiver_errors)))

if absolute_error_limit == 0.0:
    absolute_error_limit = 1e-6

for scenario_index, (
    wind_speed,
    wind_direction,
    scenario_weight,
) in enumerate(WIND_SCENARIOS["tuples"]):
    fig, ax = plt.subplots(figsize=(9, 6))

    scatter = ax.scatter(
        receiver_coords_array[
            :,
            0,
        ],
        receiver_coords_array[
            :,
            1,
        ],
        c=receiver_errors[scenario_index],
        s=55,
        vmin=(-absolute_error_limit),
        vmax=(absolute_error_limit),
    )

    ax.scatter(
        np.asarray(best_layout)[
            :,
            0,
        ],
        np.asarray(best_layout)[
            :,
            1,
        ],
        marker="x",
        s=120,
        linewidth=2,
        label="Turbines",
    )

    ax.set_aspect(
        "equal",
        adjustable="box",
    )

    ax.set_xlabel("X [m]")

    ax.set_ylabel("Y [m]")

    ax.set_title(
        (
            f"Spatial Validation Error\n"
            f"{wind_speed:.1f} m/s, "
            f"{wind_direction:.0f}°, "
            f"weight={scenario_weight:.1%}"
        )
    )

    ax.grid(
        True,
        linestyle=":",
        alpha=0.4,
    )

    ax.legend()

    colorbar = fig.colorbar(
        scatter,
        ax=ax,
    )

    colorbar.set_label("Surrogate - OpenFAST [dBA]")

    plt.tight_layout()
    plt.show()

  ## Receiver error distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    receiver_errors.ravel(),
    bins=30,
    alpha=0.8,
)

ax.axvline(
    0.0,
    linestyle="--",
    linewidth=1.5,
)

ax.axvline(
    weighted_receiver_bias,
    linestyle=":",
    linewidth=2,
    label=(f"Weighted bias " f"{weighted_receiver_bias:+.3f} dBA"),
)

ax.set_xlabel("Surrogate - OpenFAST [dBA]")

ax.set_ylabel("Count")

ax.set_title("Receiver-Level Validation Error Distribution")

ax.grid(
    axis="y",
    linestyle=":",
    alpha=0.5,
)

ax.legend()

plt.tight_layout()
plt.show()

  ## Weighted expected receiver noise maps

In [ ]:
surrogate_expected_receiver_dba = energetic_weighted_dba(
    surrogate_receiver_dba,
    SCENARIO_WEIGHTS,
    axis=0,
)

openfast_expected_receiver_dba = energetic_weighted_dba(
    openfast_receiver_dba,
    SCENARIO_WEIGHTS,
    axis=0,
)

expected_receiver_error = (
    surrogate_expected_receiver_dba - openfast_expected_receiver_dba
)

In [ ]:
expected_min = float(
    min(
        surrogate_expected_receiver_dba.min(),
        openfast_expected_receiver_dba.min(),
    )
)

expected_max = float(
    max(
        surrogate_expected_receiver_dba.max(),
        openfast_expected_receiver_dba.max(),
    )
)

for title, values in (
    (
        "Weighted Expected Receiver Noise - Surrogate",
        surrogate_expected_receiver_dba,
    ),
    (
        "Weighted Expected Receiver Noise - OpenFAST",
        openfast_expected_receiver_dba,
    ),
):
    fig, ax = plt.subplots(figsize=(9, 6))

    scatter = ax.scatter(
        receiver_coords_array[
            :,
            0,
        ],
        receiver_coords_array[
            :,
            1,
        ],
        c=values,
        s=65,
        vmin=(expected_min),
        vmax=(expected_max),
    )

    ax.scatter(
        np.asarray(best_layout)[
            :,
            0,
        ],
        np.asarray(best_layout)[
            :,
            1,
        ],
        marker="x",
        s=120,
        linewidth=2,
    )

    ax.set_aspect(
        "equal",
        adjustable="box",
    )

    ax.set_xlabel("X [m]")

    ax.set_ylabel("Y [m]")

    ax.set_title(title)

    ax.grid(
        True,
        linestyle=":",
        alpha=0.4,
    )

    colorbar = fig.colorbar(
        scatter,
        ax=ax,
    )

    colorbar.set_label("Expected dBA")

    plt.tight_layout()
    plt.show()

In [ ]:
error_limit = float(np.max(np.abs(expected_receiver_error)))

if error_limit == 0.0:
    error_limit = 1e-6

fig, ax = plt.subplots(figsize=(9, 6))

scatter = ax.scatter(
    receiver_coords_array[
        :,
        0,
    ],
    receiver_coords_array[
        :,
        1,
    ],
    c=expected_receiver_error,
    s=65,
    vmin=(-error_limit),
    vmax=(error_limit),
)

ax.scatter(
    np.asarray(best_layout)[
        :,
        0,
    ],
    np.asarray(best_layout)[
        :,
        1,
    ],
    marker="x",
    s=120,
    linewidth=2,
)

ax.set_aspect(
    "equal",
    adjustable="box",
)

ax.set_xlabel("X [m]")

ax.set_ylabel("Y [m]")

ax.set_title("Weighted Expected Noise Error\n" "Surrogate - OpenFAST")

ax.grid(
    True,
    linestyle=":",
    alpha=0.4,
)

colorbar = fig.colorbar(
    scatter,
    ax=ax,
)

colorbar.set_label("Error [dBA]")

plt.tight_layout()
plt.show()

 # Save validation results

In [ ]:
VALIDATION_OUTPUT_DIR = Path("openfast_validation_analysis")

VALIDATION_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

comparison_csv = VALIDATION_OUTPUT_DIR / "scenario_validation_comparison.csv"

comparison_df.to_csv(
    comparison_csv,
    index=False,
)

receiver_rows = []

for scenario_index, (
    speed,
    direction,
    weight,
) in enumerate(WIND_SCENARIOS["tuples"]):
    for receiver_index in range(len(receivers)):
        receiver_rows.append(
            {
                "scenario_index": (scenario_index),
                "wind_speed_mps": (speed),
                "wind_direction_deg": (direction),
                "scenario_weight": (weight),
                "receiver_index": (receiver_index),
                "receiver_x": (
                    receiver_coords_array[
                        receiver_index,
                        0,
                    ]
                ),
                "receiver_y": (
                    receiver_coords_array[
                        receiver_index,
                        1,
                    ]
                ),
                "surrogate_dba": (
                    surrogate_receiver_dba[
                        scenario_index,
                        receiver_index,
                    ]
                ),
                "openfast_dba": (
                    openfast_receiver_dba[
                        scenario_index,
                        receiver_index,
                    ]
                ),
                "error_dba": (
                    receiver_errors[
                        scenario_index,
                        receiver_index,
                    ]
                ),
            }
        )

receiver_comparison_df = pd.DataFrame(receiver_rows)

receiver_csv = VALIDATION_OUTPUT_DIR / "receiver_validation_comparison.csv"

receiver_comparison_df.to_csv(
    receiver_csv,
    index=False,
)

print(f"Saved: {comparison_csv}")

print(f"Saved: {receiver_csv}")

  # Close persistent FLORIS workers



  Run this cell when you are completely finished with FLORIS.



  If you want to rerun the GA after closing, call:



      floris_evaluator.start()

In [ ]:
floris_evaluator.close()